# ValleyFever-NewsCast

## 5. Forecasting LSTM Modeling

## Environment Setup

Run the cell below to set up the environment for either Google Colab or local execution:

In [1]:
import os
import sys

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")

    # Clone repository if in Colab
    if not os.path.exists('/content/ValleyFever-NewsCast/'):
        !git clone https://github.com/Adrian1840/ValleyFever-NewsCast
    os.chdir('/content/ValleyFever-NewsCast')

except ImportError:
    IN_COLAB = False
    print("Running locally")

# Add src directory to Python path
if 'src' not in sys.path:
    sys.path.append('src')

print(f"Current working directory: {os.getcwd()}")

Running in Google Colab
Cloning into 'ValleyFever-NewsCast'...
remote: Enumerating objects: 366, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (186/186), done.
remote: Total 366 (delta 74), reused 0 (delta 0), pack-reused 176 (from 2)
Receiving objects: 100% (366/366), 1.52 MiB | 10.81 MiB/s, done.
Resolving deltas: 100% (134/134), done.
Current working directory: /content/ValleyFever-NewsCast


## Import Libraries and Data Loading

In [2]:
pip install lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 36.7 MB/s eta 0:00:00


In [4]:
  # Basic Packages
  import numpy as np
  import pandas as pd

  #Importing LightningLSTM class and other functions
  from lstm_model import LightningLSTM, prep_county_data_gen
  from evaluation import fit_model, get_predictions_log, compute_metrics, make_dataloaders
  from plotting import plot_vf2, rename_pfi_features, plot_mean_pfi2, run_lstm_pfi_experiment

# ----Kern County----

In [6]:
kern, X_train_k, X_test_k, y_train_k, y_test_k, scaler_X_k, scaler_y_k, train_size_k, test_size_k = prep_county_data_gen(
    aggregate_path="data/external/Kern_Aggregate.csv",
    news_features_path="data/processed/news_features_unlagged.csv",
    county="Kern"
)

In [7]:
kern.head(2)

,Year-Month,VFRate,PRECIP,WIND_AvgMPH,WIND_RunMiles,AQI_PM25,AQI_PM10,EARTHQUAKE_Total,PESTICIDE_Total,Num_Articles,cv_mentions_rate,risk_mentions_rate
0,2008-10,6.476629,0.04,3.483871,83.348387,78.0,76.0,0,1392.455039,0,0.0,0.0
1,2008-11,6.110027,1.46,2.956667,70.930000,131.5,63.5,0,682.323659,2,0.0,0.0


## Kern Results

In [8]:
# Fixed model settings
seq_length_k = 6
hidden_size_k = 32
dropout_k = 0.0
lr_k = 0.0050
batch_size_k = 4
num_layers_k = 1
max_epochs_k = 100

In [9]:
train_loader_k, test_loader_k = make_dataloaders(
    X_train_k, X_test_k, y_train_k, y_test_k,
    seq_length=seq_length_k,
    batch_size=batch_size_k
)

num_features_k = X_train_k.shape[1]

kern_model = fit_model(
    num_features=num_features_k,
    seq_length=seq_length_k,
    hidden_size=hidden_size_k,
    dropout=dropout_k,
    lr=lr_k,
    train_loader=train_loader_k,
    num_layers=num_layers_k,
    max_epochs=max_epochs_k
)

kern_metrics = compute_metrics(
    model=kern_model,
    train_loader=train_loader_k,
    test_loader=test_loader_k,
    scaler_y=scaler_y_k
)

kern_metrics

INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ lstm      │ LSTM    │  5.6 K │ train │     0 │
│ 1 │ dropout   │ Dropout │      0 │ train │     0 │
│ 2 │ predictor │ Linear  │     33 │ train │     0 │
└───┴───────────┴─────────┴────────┴───────┴───────┘

Trainable params: 5.7 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 5.7 K                                                                                                
Total estimated model params size (MB): 0.023                                                                      
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO: `Trainer.fit` stopped: `max_epochs=100` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.


{'train_rmse': np.float64(3.509048858019072),
 'test_rmse': np.float64(3.495123071954333),
 'rmse_gap': np.float64(-0.013925786064739043),
 'test_pred_std': np.float64(2.4812423706985447),
 'test_actual_std': np.float64(4.662766479788639),
 'test_std_ratio': np.float64(0.5321395316393839)}

### Plotting

In [11]:
# Model Predictions and Actual Values
train_preds_k, train_actuals_k = get_predictions_log(train_loader_k, kern_model, scaler_y_k)
test_preds_k, test_actuals_k = get_predictions_log(test_loader_k, kern_model, scaler_y_k)

# --- Kern dates ---
train_dates_k = kern.iloc[seq_length_k:train_size_k]["Year-Month"]
test_dates_k = kern.iloc[train_size_k : train_size_k + len(test_actuals_k)+2]["Year-Month"]# Match the length of your predictions by starting exactly at train_size_k

In [12]:
fig_k = plot_vf2(
    "Kern",
    train_dates_k,
    train_actuals_k,
    train_preds_k,
    test_dates_k,
    test_actuals_k,
    test_preds_k
)

fig_k.show()

## ----Fresno County----

In [15]:
fresno, X_train_f, X_test_f, y_train_f, y_test_f, scaler_X_f, scaler_y_f, train_size_f, test_size_f = prep_county_data_gen(
    aggregate_path="data/external/Fresno_Aggregate.csv",
    news_features_path="data/processed/news_features_unlagged.csv",
    county="Fresno"
)

### Fresno Results

In [16]:
# Fixed model settings
seq_length_f = 6
hidden_size_f = 64
dropout_f = 0.3
lr_f = 0.0010
batch_size_f = 4
num_layers_f = 2
max_epochs_f = 100

In [27]:
train_loader_f, test_loader_f = make_dataloaders(
    X_train_f, X_test_f, y_train_f, y_test_f,
    seq_length=seq_length_f,
    batch_size=batch_size_f
)

num_features_f = X_train_f.shape[1]

fresno_model = fit_model(
    num_features=num_features_f,
    seq_length=seq_length_f,
    hidden_size=hidden_size_f,
    dropout=dropout_f,
    lr=lr_f,
    train_loader=train_loader_f,
    num_layers=num_layers_f,
    max_epochs=max_epochs_f
)

fresno_metrics = compute_metrics(
    model=fresno_model,
    train_loader=train_loader_f,
    test_loader=test_loader_f,
    scaler_y=scaler_y_f
)

fresno_metrics

INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ lstm      │ LSTM    │ 53.0 K │ train │     0 │
│ 1 │ dropout   │ Dropout │      0 │ train │     0 │
│ 2 │ predictor │ Linear  │     65 │ train │     0 │
└───┴───────────┴─────────┴────────┴───────┴───────┘

Trainable params: 53.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 53.1 K                                                                                               
Total estimated model params size (MB): 0.212                                                                      
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: FutureWarning:

`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: `Trainer.fit` stopped: `max_epochs=100` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.


{'train_rmse': np.float64(1.1311318256974034),
 'test_rmse': np.float64(1.228138976875302),
 'rmse_gap': np.float64(0.09700715117789849),
 'test_pred_std': np.float64(0.15924516963576307),
 'test_actual_std': np.float64(1.2611728899944865),
 'test_std_ratio': np.float64(0.12626751724457005)}

### Plotting

In [19]:
#Fresno predictions and Actuals
train_preds_f, train_actuals_f = get_predictions_log(train_loader_f, fresno_model, scaler_y_f)
test_preds_f, test_actuals_f = get_predictions_log(test_loader_f, fresno_model, scaler_y_f)


# --- Fresno dates ---
train_dates_f = fresno.iloc[seq_length_f:train_size_f]["Year-Month"]
test_dates_f = fresno.iloc[train_size_f : train_size_f + len(test_actuals_f)+2]["Year-Month"] #test dates were misaligned forward by 6 months

In [20]:
fig_f = plot_vf2(
    "Fresno",
    train_dates_f,
    train_actuals_f,
    train_preds_f,
    test_dates_f,
    test_actuals_f,
    test_preds_f
)

fig_f.show()

## Average PFI Scores across 100 Runs

In [21]:
#Gets feature names (env variables and num_articles, NOT VFRate)
feature_names_k = kern.columns[2:]
feature_names_f = fresno.columns[2:]

## Saving PFI Scores across 20 Runs

In [22]:
#Gets feature names (env variables and num_articles, NOT VFRate)
feature_names_k = kern.columns[2:]
feature_names_f = fresno.columns[2:]

In [ ]:
kern_pfis = run_lstm_pfi_experiment(
    county_name="Kern",
    num_features=X_train_k.shape[1],
    seq_length=seq_length_k,
    hidden_size=hidden_size_k,
    dropout=dropout_k,
    lr=lr_k,
    num_layers=num_layers_k,
    train_loader=train_loader_k,
    test_loader=test_loader_k,
    X_test=X_test_k,
    y_test=y_test_k,
    scaler_y=scaler_y_k,
    feature_names=feature_names_k,
    num_runs=100,
    max_epochs=max_epochs_k
)

Kern Run 1/2


INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ lstm      │ LSTM    │  5.6 K │ train │     0 │
│ 1 │ dropout   │ Dropout │      0 │ train │     0 │
│ 2 │ predictor │ Linear  │     33 │ train │     0 │
└───┴───────────┴─────────┴────────┴───────┴───────┘

Trainable params: 5.7 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 5.7 K                                                                                                
Total estimated model params size (MB): 0.023                                                                      
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
INFO:lightning.pytorch.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 630, in _fit_impl
    self._run(model, ckpt_path=ckpt_path, weights_only=weights_only)
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 1079, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 1123, in _run_stage
    self.fit_loop.run()
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py", line 217, in run
    self.advance()
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py", line 469, in advance
    self.epoch_loop.run(self._data_fetcher)
  File

TypeError: object of type 'NoneType' has no len()

In [ ]:
fresno_pfis = run_lstm_pfi_experiment(
    county_name="Fresno",
    num_features=X_train_f.shape[1],
    seq_length=seq_length_f,
    hidden_size=hidden_size_f,
    dropout=dropout_f,
    lr=lr_f,
    num_layers=num_layers_f,
    train_loader=train_loader_f,
    test_loader=test_loader_f,
    X_test=X_test_f,
    y_test=y_test_f,
    scaler_y=scaler_y_f,
    feature_names=feature_names_f,
    num_runs=100,
    max_epochs=max_epochs_f
)

INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Fresno Run 1/100


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ lstm      │ LSTM    │ 53.0 K │ train │     0 │
│ 1 │ dropout   │ Dropout │      0 │ train │     0 │
│ 2 │ predictor │ Linear  │     65 │ train │     0 │
└───┴───────────┴─────────┴────────┴───────┴───────┘

Trainable params: 53.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 53.1 K                                                                                               
Total estimated model params size (MB): 0.212                                                                      
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO: `Trainer.fit` stopped: `max_epochs=100` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to 

...

Fresno Run 100/100


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ lstm      │ LSTM    │ 53.0 K │ train │     0 │
│ 1 │ dropout   │ Dropout │      0 │ train │     0 │
│ 2 │ predictor │ Linear  │     65 │ train │     0 │
└───┴───────────┴─────────┴────────┴───────┴───────┘

Trainable params: 53.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 53.1 K                                                                                               
Total estimated model params size (MB): 0.212                                                                      
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO: `Trainer.fit` stopped: `max_epochs=100` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.


Saved: fresno_pfis_100.csv


### Importing PFI Scores

In [23]:
CSV_files = "/content/ValleyFever-NewsCast/results/pfi/"
fresPFIpath = "fresno_pfis_100.csv"
kernPFIpath = "kern_pfis_100.csv"


fresno_pfis = pd.read_csv(CSV_files + fresPFIpath)
kern_pfis = pd.read_csv(CSV_files + kernPFIpath)

## Plotting PFI Scores

In [24]:
# Define the mapping for your columns
rename_dict = {
    "cv_mentions_rate": "Cen_Valley_Mentions_Rate",
    "risk_mentions_rate": "Risk_Mentions_Rate",
    }

# Apply the rename to the columns
kern_pfis["feature"] = kern_pfis["feature"].replace(rename_dict)
fresno_pfis["feature"] = fresno_pfis["feature"].replace(rename_dict)

In [25]:
plot_mean_pfi2(fresno_pfis, "Fresno")

In [26]:
plot_mean_pfi2(kern_pfis, "Kern", "darkgreen")